# 01 - Análise Exploratória de Dados

**Exploração dos dados** · Dados Hospitalares - Diabetes 130-US Hospitals for Years 1999-2008

---

Este notebook percorre o **roteiro de oito etapas** da análise exploratória sobre a
base bruta de Diabetes 130-US Hospitals, usando o dicionário de dados montado em
[`00-dicionario-dados.ipynb`](00-dicionario-dados.ipynb) como mapa de tipos:

1. **Distribuição dos atributos** — o comportamento de cada característica analisada.
2. **Análise univariada** — distribuição de frequências e medidas de localização,
   dispersão, assimetria e curtose, uma variável por vez.
3. **Análise bivariada** — se a variação de uma característica é acompanhada pela
   variação de outra.
4. **Análise multivariada** — padrões e relações entre várias variáveis ao mesmo tempo.
5. **Valores ausentes** — onde falta dado e quanto essa falta pesa.
6. **Valores discrepantes** — registros cujo comportamento foge do esperado.
7. **Correlação** — qualquer relação estatística entre duas ou mais variáveis.
8. **Seleção de características** — a combinação de variáveis relevante para o
   aprendizado que vem depois.

Cada etapa grava as figuras em `docs/imagens/eda/` e a tabela que originou cada
figura em `reports/tables/eda-exploratoria/` — a regra do projeto é que
**nenhuma figura existe sem a tabela que a produziu**.


In [48]:
%matplotlib inline

# Importações
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Configurações de visualização
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

In [49]:
# Funções para aplicar as escalas

def carregar_escalas(caminho):
    escala = pd.read_csv(
        caminho,
        sep=None,
        engine='python',
        dtype={'coluna': str, 'nivel': str},
        encoding='utf-8-sig'
    )

    escala.columns = escala.columns.str.strip().str.lower()

    esperadas = {'coluna', 'nivel', 'ordem'}
    faltando = esperadas - set(escala.columns)

    if faltando:
        raise ValueError(
            f'Colunas ausentes em {caminho}: {faltando}. '
            f'Colunas encontradas: {escala.columns.tolist()}'
        )

    escala = escala.sort_values(['coluna', 'ordem'])

    return escala.groupby('coluna')['nivel'].apply(list).to_dict()


def aplicar_escalas(dados, escalas):
    for coluna, niveis in escalas.items():

        if coluna not in dados.columns:
            print(
                f'Aviso: a coluna "{coluna}" está na escala, '
                'mas não existe no dataset.'
            )
            continue

        # Identifica valores que não pertencem à escala.
        valores_validos = dados[coluna].dropna()
        valores_validos = valores_validos[
            valores_validos.astype(str) != '?'
        ]

        fora_da_escala = (
            set(valores_validos.astype(str)) - set(map(str, niveis))
        )

        if fora_da_escala:
            raise ValueError(
                f'Valores fora da escala em "{coluna}": '
                f'{fora_da_escala}'
            )

        # Converte '?' em ausente somente na variável ordinal transformada.
        # A base original deve ser preservada.
        serie = dados[coluna].replace('?', pd.NA)

        if pd.api.types.is_numeric_dtype(dados[coluna]):
            niveis = pd.to_numeric(pd.Series(niveis)).tolist()

        dados[coluna] = pd.Categorical(
            serie,
            categories=niveis,
            ordered=True
        )

    return dados

In [50]:
# Carregando base de dados
dados = pd.read_csv('../data/raw/diabetic_data.csv')

# Carregando dicionário dos dados
dicionario = pd.read_csv('../data/processed/dicionario.csv', sep=';')

# Carregando escalas
escalas = carregar_escalas(
    caminho='../data/processed/escala.csv'
)

# Criando cópia para análise
dados = dados.copy()

In [51]:
# Verificando estrutura inicial
print(f'Dimensões da base: {dados.shape}')
print(f'Quantidade de pacientes: {dados["patient_nbr"].nunique():,}')
print(f'Quantidade de atendimentos: {dados["encounter_id"].nunique():,}')

display(dados.head())

Dimensões da base: (101766, 50)
Quantidade de pacientes: 71,518
Quantidade de atendimentos: 101,766


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [52]:
# Fazendo o mesmo processo de renomar as colunas do notebook 00
# Renomear todas as colunas (ingles para)
renomear_colunas = {
    'encounter_id': 'id_atendimento',
    'patient_nbr': 'id_paciente',
    'race': 'raca',
    'gender': 'sexo',
    'age': 'faixa_etaria',
    'weight': 'peso',
    'admission_type_id': 'id_tipo_admissao',
    'discharge_disposition_id': 'id_destino_alta',
    'admission_source_id': 'id_origem_admissao',
    'time_in_hospital': 'dias_internacao',
    'payer_code': 'codigo_fonte_pagadora',
    'medical_specialty': 'especialidade_medica',
    'num_lab_procedures': 'qtd_procedimentos_laboratoriais',
    'num_procedures': 'qtd_procedimentos',
    'num_medications': 'qtd_medicamentos',
    'number_outpatient': 'qtd_atendimentos_ambulatoriais',
    'number_emergency': 'qtd_atendimentos_emergenciais',
    'number_inpatient': 'qtd_internacoes_anteriores',
    'diag_1': 'diagnostico_principal',
    'diag_2': 'diagnostico_secundario_1',
    'diag_3': 'diagnostico_secundario_2',
    'number_diagnoses': 'qtd_diagnosticos',
    'max_glu_serum': 'resultado_maximo_glicemia',
    'A1Cresult': 'resultado_hemoglobina_glicada',
    'metformin': 'metformina',
    'repaglinide': 'repaglinida',
    'nateglinide': 'nateglinida',
    'chlorpropamide': 'clorpropamida',
    'glimepiride': 'glimepirida',
    'acetohexamide': 'acetohexamida',
    'glipizide': 'glipizida',
    'glyburide': 'gliburida',
    'tolbutamide': 'tolbutamida',
    'pioglitazone': 'pioglitazona',
    'rosiglitazone': 'rosiglitazona',
    'acarbose': 'acarbose',
    'miglitol': 'miglitol',
    'troglitazone': 'troglitazona',
    'tolazamide': 'tolazamida',
    'examide': 'examida',
    'citoglipton': 'citogliptona',
    'insulin': 'insulina',
    'glyburide-metformin': 'gliburida_metformina',
    'glipizide-metformin': 'glipizida_metformina',
    'glimepiride-pioglitazone': 'glimepirida_pioglitazona',
    'metformin-rosiglitazone': 'metformina_rosiglitazona',
    'metformin-pioglitazone': 'metformina_pioglitazona',
    'change': 'alteracao_medicacao',
    'diabetesMed': 'uso_medicacao_diabetes',
    'readmitted': 'situacao_reinternacao'
}

dados.rename(columns=renomear_colunas, inplace=True)

In [53]:
# Casos específicos da base adicional (admissão, alta e origem)
MAP_TIPO_ADMISSAO = {
    1: 'Emergency',
    2: 'Urgent',
    3: 'Elective',
    4: 'Newborn',
    5: 'Not Available',
    6: None,
    7: 'Trauma Center',
    8: 'Not Mapped'
}

MAP_DESTINO_ALTA = {
    1: 'Discharged to home',
    2: 'Transferred to short term hospital',
    3: 'Transferred to SNF',
    4: 'Transferred to ICF',
    5: 'Transferred to another inpatient care institution',
    6: 'Discharged to home with home health service',
    7: 'Left AMA',
    8: 'Discharged to home under Home IV provider',
    9: 'Admitted as inpatient to this hospital',
    10: 'Neonate transferred for neonatal aftercare',
    11: 'Expired',
    12: 'Still patient or expected outpatient return',
    13: 'Hospice / home',
    14: 'Hospice / medical facility',
    15: 'Transferred to Medicare approved swing bed',
    16: 'Referred to another institution for outpatient services',
    17: 'Referred to this institution for outpatient services',
    18: None,
    19: 'Expired at home',
    20: 'Expired in a medical facility',
    21: 'Expired, place unknown',
    22: 'Transferred to rehabilitation facility',
    23: 'Transferred to long term care hospital',
    24: 'Transferred to Medicaid nursing facility',
    25: 'Not Mapped',
    26: 'Unknown/Invalid',
    27: 'Transferred to federal health care facility',
    28: 'Transferred to psychiatric hospital',
    29: 'Transferred to Critical Access Hospital',
    30: 'Transferred to another health care institution'
}

MAP_ORIGEM_ADMISSAO = {
    1: 'Physician Referral',
    2: 'Clinic Referral',
    3: 'HMO Referral',
    4: 'Transfer from a hospital',
    5: 'Transfer from SNF',
    6: 'Transfer from another health care facility',
    7: 'Emergency Room',
    8: 'Court/Law Enforcement',
    9: 'Not Available',
    10: 'Transfer from critical access hospital',
    11: 'Normal Delivery',
    12: 'Premature Delivery',
    13: 'Sick Baby',
    14: 'Extramural Birth',
    15: 'Not Available',
    17: None,
    18: 'Transfer from another Home Health Agency',
    19: 'Readmission to same Home Health Agency',
    20: 'Not Mapped',
    21: 'Unknown/Invalid',
    22: 'Transfer from hospital inpatient/same facility',
    23: 'Born inside this hospital',
    24: 'Born outside this hospital',
    25: 'Transfer from Ambulatory Surgery Center',
    26: 'Transfer from Hospice'
}

In [54]:
MAPEAMENTOS = {
    'id_tipo_admissao': MAP_TIPO_ADMISSAO,
    'id_destino_alta': MAP_DESTINO_ALTA,
    'id_origem_admissao': MAP_ORIGEM_ADMISSAO
}

In [55]:
# Renomeando as colunas
dados.rename(columns=renomear_colunas, inplace=True)

# Criando variáveis descritivas a partir dos códigos
dados['tipo_admissao_desc'] = dados['id_tipo_admissao'].map(MAP_TIPO_ADMISSAO)
dados['destino_alta_desc'] = dados['id_destino_alta'].map(MAP_DESTINO_ALTA)
dados['origem_admissao_desc'] = dados['id_origem_admissao'].map(MAP_ORIGEM_ADMISSAO)

In [56]:
# selecionando as mesmas colunas do dicionário de dados
dados = dados.loc[:,dicionario.variavel]

In [57]:
dicionario

,variavel,descricao,valores,tipo,qtd_categorias,qtd_nan,qtd_interrogacao,mapeamento
0,id_atendimento,Id atendimento,"[2278392, 149190, 64410, 500364, 16680, 35754,...",Identificador,101766,0,0,NaN
1,id_paciente,Id paciente,"[8222157, 55629189, 86047875, 82442376, 425192...",Identificador,71518,0,0,NaN
2,raca,Raca,"['Caucasian', 'AfricanAmerican', '?', 'Other',...",Nominal,6,0,2273,NaN
3,sexo,Sexo,"['Female', 'Male', 'Unknown/Invalid']",Nominal,3,0,0,NaN
4,faixa_etaria,Faixa etaria,"['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[...",Ordinal,10,0,0,NaN
5,peso,Peso,"['?', '[75-100)', '[50-75)', '[0-25)', '[100-1...",Ordinal,10,0,98569,NaN
6,id_tipo_admissao,Id tipo admissao,"[6, 1, 2, 3, 4, 5, 8, 7]",Nominal,8,0,0,"{1: 'Emergency', 2: 'Urgent', 3: 'Elective', 4..."
7,id_destino_alta,Id destino alta,"[25, 1, 3, 6, 2, 5, 11, 7, 10, 4, 14, 18, 8, 1...",Nominal,26,0,0,"{1: 'Discharged to home', 2: 'Transferred to s..."
8,id_origem_admissao,Id origem admissao,"[1, 7, 2, 4, 5, 6, 20, 3, 17, 8, 9, 14, 10, 22...",Nominal,17,0,0,"{1: 'Physician Referral', 2: 'Clinic Referral'..."
9,dias_internacao,Dias internacao,"[1, 3, 2, 4, 5, 13, 12, 9, 7, 10, 6, 11, 8, 14]",Discreta,14,0,0,NaN


In [58]:
print(f'Dimensões da base: {dados.shape}')
print(f'Dimensões do dicionário: {dicionario.shape}')

Dimensões da base: (101766, 53)
Dimensões do dicionário: (53, 8)


## Análises iniciais

In [66]:
print('Dimensões:', dados.shape)
dados.info()
display(dados.describe(include='all').T)

Dimensões: (101766, 53)
<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 53 columns):
 #   Column                           Non-Null Count   Dtype
---  ------                           --------------   -----
 0   id_atendimento                   101766 non-null  int64
 1   id_paciente                      101766 non-null  int64
 2   raca                             101766 non-null  str  
 3   sexo                             101766 non-null  str  
 4   faixa_etaria                     101766 non-null  str  
 5   peso                             101766 non-null  str  
 6   id_tipo_admissao                 101766 non-null  int64
 7   id_destino_alta                  101766 non-null  int64
 8   id_origem_admissao               101766 non-null  int64
 9   dias_internacao                  101766 non-null  int64
 10  codigo_fonte_pagadora            101766 non-null  str  
 11  especialidade_medica             101766 non-null  str  
 12  qtd_procedimentos

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_atendimento,101766.0,NaN,NaN,NaN,165201645.622978,102640295.983457,12522.0,84961194.0,152388987.0,230270887.5,443867222.0
id_paciente,101766.0,NaN,NaN,NaN,54330400.694947,38696359.346534,135.0,23413221.0,45505143.0,87545949.75,189502619.0
raca,101766,6,Caucasian,76099,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sexo,101766,3,Female,54708,NaN,NaN,NaN,NaN,NaN,NaN,NaN
faixa_etaria,101766,10,[70-80),26068,NaN,NaN,NaN,NaN,NaN,NaN,NaN
peso,101766,10,?,98569,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_tipo_admissao,101766.0,NaN,NaN,NaN,2.024006,1.445403,1.0,1.0,1.0,3.0,8.0
id_destino_alta,101766.0,NaN,NaN,NaN,3.715642,5.280166,1.0,1.0,1.0,4.0,28.0
id_origem_admissao,101766.0,NaN,NaN,NaN,5.754437,4.064081,1.0,1.0,7.0,7.0,25.0
dias_internacao,101766.0,NaN,NaN,NaN,4.395987,2.985108,1.0,2.0,4.0,6.0,14.0


In [67]:
# Quantidade por tipo
dicionario.tipo.value_counts()

tipo
Nominal          36
Discreta          8
Ordinal           4
Derivada          3
Identificador     2
Name: count, dtype: int64

- 36 variáveis nominais, incluindo diagnósticos, medicamentos e características demográficas.
- 8 variáveis discretas, principalmente contagens e quantidades.
- 4 variáveis ordinais, com hierarquia definida.
- 3 variáveis derivadas do tipo de admissao, de alta e origem de admissao.
- 2 identificadores, que não devem entrar diretamente no algoritmo.

In [68]:
# Estatísticas descritivas
display(dados.describe(include='all').T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_atendimento,101766.0,NaN,NaN,NaN,165201645.622978,102640295.983457,12522.0,84961194.0,152388987.0,230270887.5,443867222.0
id_paciente,101766.0,NaN,NaN,NaN,54330400.694947,38696359.346534,135.0,23413221.0,45505143.0,87545949.75,189502619.0
raca,101766,6,Caucasian,76099,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sexo,101766,3,Female,54708,NaN,NaN,NaN,NaN,NaN,NaN,NaN
faixa_etaria,101766,10,[70-80),26068,NaN,NaN,NaN,NaN,NaN,NaN,NaN
peso,101766,10,?,98569,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_tipo_admissao,101766.0,NaN,NaN,NaN,2.024006,1.445403,1.0,1.0,1.0,3.0,8.0
id_destino_alta,101766.0,NaN,NaN,NaN,3.715642,5.280166,1.0,1.0,1.0,4.0,28.0
id_origem_admissao,101766.0,NaN,NaN,NaN,5.754437,4.064081,1.0,1.0,7.0,7.0,25.0
dias_internacao,101766.0,NaN,NaN,NaN,4.395987,2.985108,1.0,2.0,4.0,6.0,14.0


### Achados pós-describe (precisamos ESTUDAR isso para podermos decidir o que fazer na engenharia de features)

#### **1. Variáveis categóricas e cardinalidade**
|Variável|Categorias distintas|
|---|---|
|diagnostico_principal|717|
|diagnostico_secundario_1|749|
|diagnostico_secundario_2|790|
|especialidade_medica|73|
|codigo_fonte_pagadora|18|

#### **2. Ausências e marcadores especiais**

- NaN: ausência efetiva de valor, como nos resultados de glicemia e hemoglobina glicada.
- ?: marcador textual utilizado em determinadas variáveis.
- Códigos sem correspondência nos mapeamentos: podem produzir ausências nas variáveis derivadas.

#### **3. Variáveis de medicamentos**

|Variável|Categorias distintas|
|---|---|
|examida|1|
|citogliptona|1|
|acetohexamida|2|
|tolbutamida|2|
|troglitazona|2|

- examida e citogliptona apresentam apenas a categoria No em todos os registros.
- São variáveis constantes, sem capacidade de discriminar os registros na base. Mais adiante, devemos excluir dos atributos?
- Vamos primeiro precisamos documentar e compreender melhor.

#### **4. Variáveis numéricas**

- qtd_atendimentos_ambulatoriais: 75% dos registros apresentam valor zero.
- qtd_atendimentos_emergenciais: também apresenta mediana e terceiro quartil iguais a zero.
- qtd_internacoes_anteriores: mediana igual a zero.
- qtd_medicamentos: média de aproximadamente 16, mas máximo de 81.

Isso é uma assimetria concordm? Tem uma concentração de valores baixos em algumas variáveis. Precisaremos observar essas distribuições graficamente.

#### **5. Inconsistências?**

- Verificando contagens e de valores ausentes.



In [ ]:
# Conferindo ausências nas variáveis derivadas
colunas_derivadas = [
    'tipo_admissao_desc',
    'destino_alta_desc',
    'origem_admissao_desc'
]

display(dados[colunas_derivadas].isna().sum())

tipo_admissao_desc      5291
destino_alta_desc       3691
origem_admissao_desc    6781
dtype: int64

In [ ]:
# Conferindo códigos sem correspondência
for coluna in [
    'id_tipo_admissao',
    'id_destino_alta',
    'id_origem_admissao'
]:
    print(f'\n{coluna}')
    print(dados[coluna].value_counts(dropna=False).sort_index())


id_tipo_admissao
id_tipo_admissao
1    53990
2    18480
3    18869
4       10
5     4785
6     5291
7       21
8      320
Name: count, dtype: int64

id_destino_alta
id_destino_alta
1     60234
2      2128
3     13954
4       815
5      1184
6     12902
7       623
8       108
9        21
10        6
11     1642
12        3
13      399
14      372
15       63
16       11
17       14
18     3691
19        8
20        2
22     1993
23      412
24       48
25      989
27        5
28      139
Name: count, dtype: int64

id_origem_admissao
id_origem_admissao
1     29565
2      1104
3       187
4      3187
5       855
6      2264
7     57494
8        16
9       125
10        8
11        2
13        1
14        2
17     6781
20      161
22       12
25        2
Name: count, dtype: int64


#### 6. Questões identificadas até aqui

- alta cardinalidade dos diagnósticos 
- variáveis com muitos valores ausentes 
- variáveis constantes ou muito desbalanceadas

## Análise univariada

### Características nominais

|Aspecto|O que investigar|Relevância para o projeto|
|---|---|---|
|Concentração|Uma categoria representa quase todos os registros?|Pode indicar variável pouco discriminativa.
|Desbalanceamento|Uma categoria é muito mais frequente que as demais?|Pode influenciar a formação dos grupos.
|Cardinalidade|Existem muitas categorias diferentes?|Pode exigir tratamento específico na codificação.
|Categorias desconhecidas|Aparecem ?, Unknown/Invalid ou Not Mapped?|Exige decisões de tratamento.
|Categorias raras|Existem categorias com pouquíssimos registros?|Podem gerar ruído ou grupos muito pequenos.

In [72]:
# Obter uma lista das variáveis nominais
nominal_columns = (
    dicionario
    .query('tipo == "Nominal"')
    .variavel
    .to_list()
)

In [75]:
if nominal_columns:
    for column in nominal_columns:

        # Calculando frequências
        frequencias = df[column].value_counts()
        top_10 = frequencias.head(10)

        # Total de registros com valores contabilizados
        total = frequencias.sum()

        # Gerando gráfico
        ax = sns.barplot(
            x=top_10.values,
            y=top_10.index,
            color='#183EFF',
            alpha=1
        )

        # Configurando estilo e tamanho
        sns.set_style('whitegrid')
        ax.figure.set_size_inches(25, 8)

        # Exibindo frequências absolutas e relativas
        for container in ax.containers:
            ax.bar_label(
                container,
                label_type='edge',
                labels=[
                    f'{valor:.0f}\n{valor/total:.2%}'
                    for valor in top_10.values
                ]
            )

        # Linha de referência: média das dez maiores frequências
        ax.axvline(
            top_10.mean(),
            color='#FFD000',
            clip_on=False,
            linestyle='dashed'
        )

        # Configurações visuais
        sns.despine(left=False)
        ax.axvline(0, color='k', clip_on=False)
        plt.xlabel('Quantidade de registros')
        plt.title(
            'As categorias mais frequentes da característica ' + column
        )

        # Salvando figura
        fig = ax.get_figure()
        fig.savefig(
            f'../docs/imagens/eda/univariada-nominal-{column}.png',
            bbox_inches='tight'
        )
        plt.close(fig)

**Considerando a estrutura da nossa base, temos de dar atenção a estas:**
- **raca e sexo**: composição demográfica dos atendimentos.
- **especialidade_medica**: diversidade e concentração das especialidades.
- **diagnostico_principal, diagnostico_secundario_1 e diagnostico_secundario_2**: cardinalidade e concentração dos diagnósticos.
- **id_tipo_admissao, id_destino_alta e id_origem_admissao**: distribuição dos tipos de admissão, destinos de alta e origens.
- **metformina, insulina e demais medicamentos**: frequência de utilização e alterações na medicação.
- **situacao_reinternacao**: distribuição da variável que poderá ser utilizada posteriormente para caracterização dos agrupamentos, mas não como atributo do clustering.

In [76]:
# Relatório para ver de forma vertical os dados de cada categotia
for column in nominal_columns:
    print(f"\n{'=' * 65}")
    print(f"VARIÁVEL: {column}")
    print(f"{'=' * 65}")

    frequencias = dados[column].value_counts(dropna=False)

    print(f"Categorias distintas: {dados[column].nunique()}")
    print(f"Registros válidos: {dados[column].notna().sum()}")
    print(f"Valores ausentes: {dados[column].isna().sum()}")

    print("\n10 categorias mais frequentes:")

    resumo = pd.DataFrame({
        'frequencia': frequencias.head(10),
        'percentual_total': (
            frequencias.head(10) / len(dados) * 100
        ).round(2)
    })

    print(resumo.to_string())


VARIÁVEL: raca
Categorias distintas: 6
Registros válidos: 101766
Valores ausentes: 0

10 categorias mais frequentes:
                 frequencia  percentual_total
raca                                         
Caucasian             76099             74.78
AfricanAmerican       19210             18.88
?                      2273              2.23
Hispanic               2037              2.00
Other                  1506              1.48
Asian                   641              0.63

VARIÁVEL: sexo
Categorias distintas: 3
Registros válidos: 101766
Valores ausentes: 0

10 categorias mais frequentes:
                 frequencia  percentual_total
sexo                                         
Female                54708             53.76
Male                  47055             46.24
Unknown/Invalid           3              0.00

VARIÁVEL: id_tipo_admissao
Categorias distintas: 8
Registros válidos: 101766
Valores ausentes: 0

10 categorias mais frequentes:
                  frequencia  percen

In [77]:
# Identificando ocorrências do marcador "?"
for coluna in dados.columns:
    quantidade = (dados[coluna] == '?').sum()

    if quantidade > 0:
        percentual = quantidade / len(dados) * 100

        print(
            f'{coluna}: {quantidade:,} '
            f'({percentual:.2f}%)'
        )

raca: 2,273 (2.23%)
peso: 98,569 (96.86%)
codigo_fonte_pagadora: 40,256 (39.56%)
especialidade_medica: 49,949 (49.08%)
diagnostico_principal: 21 (0.02%)
diagnostico_secundario_1: 358 (0.35%)
diagnostico_secundario_2: 1,423 (1.40%)


- Especialidade médica: não há informação sobre a especialidade associada ao atendimento. Isso não significa necessariamente que o atendimento não tenha sido realizado por um especialista.
- Fonte pagadora: não há informação identificada sobre a fonte pagadora.
- Raça: a informação não foi registrada ou disponibilizada.

> Acho interessante ainda não tratar isso. Devemos continuar a análise e na etapa de pré-processamento, tomarmos uma decisão sobre o que realizar com esses valores que não estão ausentes mas estão inespecíficaos.

## Análise bivariada